# Layer 4 of 4 - Validation results

Consumes the selected n-gram result from Layer 2 and completed transformer run records from Layer 3. It writes a validation comparison table and training-curve plot under `results/`.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


current = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in (current, *current.parents) if (path / ".git").exists()), current)
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
RUNS_DIR = RESULTS_DIR / "runs"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
for directory in (DATA_DIR, RESULTS_DIR, RUNS_DIR, CHECKPOINT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
if not RUNS_DIR.exists():
    raise FileNotFoundError("Run layers/03_training/03_train_transformer.ipynb first.")


## 9. Validation results

Every finished run's selected checkpoint, scored with the fixed-target metric on validation.

In [ ]:
runs = [json.loads(path.read_text(encoding="utf-8")) for path in sorted(RUNS_DIR.glob("*.json"))]

rows = []
for run in runs:
    config = run["config"]
    rows.append({
        "model": "transformer",
        "context_len": config["context_len"],
        "seed": config["seed"],
        "best_step": run["best_step"],
        "train_minutes": round(run["train_seconds"] / 60, 1),
        "val_bpc": run["val_bpc"],
    })

ngram_path = RESULTS_DIR / "ngram.json"
if not ngram_path.exists():
    raise FileNotFoundError("Run layers/02_ngram/02_ngram_reference.ipynb first.")
best_ngram = json.loads(ngram_path.read_text(encoding="utf-8"))["best"]
rows.append({"model": f"{best_ngram['order']}-gram (d={best_ngram['discount']})", "val_bpc": best_ngram["val_bpc"]})

results = pd.DataFrame(rows)
results.to_csv(RESULTS_DIR / "validation_results.csv", index=False)
results


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for run in runs:
    steps = [point["step"] for point in run["history"]]
    ax.plot(steps, [point["train_loss"] for point in run["history"]], label=f"{run['config']['context_len']} train")
    ax.plot(steps, [point["val_loss"] for point in run["history"]], "--", label=f"{run['config']['context_len']} validation")
ax.set_xlabel("training step")
ax.set_ylabel("cross-entropy (nats per character)")
ax.set_title("Training curves by context length")
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "training_curves.png", dpi=150)